In [1]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
llm = ChatOllama(model="gemma4:e2b", base_url="http://127.0.0.1:11434")

llm.invoke([HumanMessage("잘 지냈어?")])

AIMessage(content='응, 잘 지냈어! 😊 너는 잘 지냈어?', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-22T06:23:18.5829328Z', 'done': True, 'done_reason': 'stop', 'total_duration': 32541635200, 'load_duration': 28804475000, 'prompt_eval_count': 21, 'prompt_eval_duration': 235334000, 'eval_count': 315, 'eval_duration': 3494413000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a0c7c7-f714-79b0-b8c1-45f42438639a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 21, 'output_tokens': 315, 'total_tokens': 336})

In [2]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time


In [3]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [4]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

# (6) 생성된 llm 답변 출력
print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-22T06:23:31.8047108Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3698976100, 'load_duration': 6069800, 'prompt_eval_count': 157, 'prompt_eval_duration': 813266000, 'eval_count': 291, 'eval_duration': 2866110000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a0c7c8-9b65-72f0-b5dd-8932a0a185c5-0', tool_calls=[{'name': 'get_current_time', 'args': {'location': '부산', 'timezone': 'Asia/Seoul'}, 'id': '2c445860-653d-48ef-8ce9-5c8963a996f1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 157, 'output_tokens': 291, 'total_tokens': 448})]


In [5]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'location': '부산', 'timezone': 'Asia/Seoul'}
Asia/Seoul (부산) 현재시각 2026-09-22 15:23:31 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-22T06:23:31.8047108Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3698976100, 'load_duration': 6069800, 'prompt_eval_count': 157, 'prompt_eval_duration': 813266000, 'eval_count': 291, 'eval_duration': 2866110000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a0c7c8-9b65-72f0-b5dd-8932a0a185c5-0', tool_calls=[{'name': 'get_current_time', 'args': {'location': '부산', 'timezone': 'Asia/Seoul'}, 'id': '2c445860-653d-48ef-8ce9-5c8963a996f1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 157, 'output_tokens': 291, 'total_tokens': 448}),
 ToolMessage(content='Asia/Seoul (부산) 현재시각 2026-09-22 15:23:31 ', name='get_

In [6]:
llm_with_tools.invoke(messages)

AIMessage(content='부산은 현재 **2026년 9월 22일 15시 23분 31초**입니다.', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-22T06:23:33.4716785Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1521039900, 'load_duration': 8353800, 'prompt_eval_count': 225, 'prompt_eval_duration': 75809000, 'eval_count': 150, 'eval_duration': 1426946000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a0c7c8-aa6d-7042-ab4a-d77459930f35-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 225, 'output_tokens': 150, 'total_tokens': 375})